<a href="https://colab.research.google.com/github/yutaota/intro_to_causal_inference/blob/main/Ex04_Regression_with_Covariate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 共通設定：X, A の生成

In [28]:
# 利用するライブラリのインポート
import numpy as np
import pandas as pd
import statsmodels.api as sm
import os

# シードを設定して再現性を確保
np.random.seed(1)

# サンプル数
N = 2000

# 共変量 (X)
# 共変量は互いに独立な標準正規分布から生成
# X1:性別、X2:年齢, X3:重症度, X4:家族の経済状況, X5:遺伝的体質
# X1、X2, X3はYとAに関連し、X4, X5はYのみに関連する。
X = np.random.randn(N, 5)
X[:, 0] = np.random.exponential(1, N) - 1  # 非対称、平均0付近

# 処置変数 (A): X1:性別、X2:年齢, X3:重症度に依存する。
prob_t = 1 / (1 + np.exp(-(0.5 * X[:, 0] + 0.8 * X[:, 1]+ 0.3 * X[:, 2])))
#prob_t = 1 / (1 + np.exp(-(0.5 * X[:, 0] + 0.8 * X[:, 1]+ 0.3 * X[:, 2]+ 1 * X[:, 3]))) # X4:家族の経済状況も処置に影響
A = np.random.binomial(1, prob_t)
print("処置人数:", sum(A))

処置人数: 1031


# シナリオ1. 効果の同質性

In [29]:
# アウトカム変数 (Y)
# アウトカムは処置AとすべてのXに「線形に」依存し、処置効果の異質性はなし
true_ATE = beta = 2
u = 0.5 * X[:, 3] - 0.2 * X[:, 4] + np.random.rand(N)
Y = beta * A + 1.2 * X[:, 0] + 0.6 * X[:, 1] - 0.3 * X[:, 2]  + u
print(f"真のATE: {true_ATE:.3f}")

data = pd.DataFrame(X[:, 0:3], columns=["X1", "X2", "X3"])
data['A'] = A
data['Y'] = Y


# 1. 単回帰分析を実行（ATEにバイアスが生まれるはず）
X_reg = sm.add_constant(data[['A']]) # 定数項を追加
y_reg = data['Y']

model = sm.OLS(y_reg, X_reg)
results = model.fit()

# 分析結果を表示
print(results.summary())

# 2. 共変量を加えた回帰分析を実行
X_reg = sm.add_constant(data[['A','X1', 'X2', 'X3']]) # 定数項を追加
# X_reg = sm.add_constant(data[['A','X1', 'X2', 'X3', 'X4', 'X5']]) # 定数項を追加
y_reg = data['Y']

model = sm.OLS(y_reg, X_reg)
results = model.fit()

# 分析結果を表示
print(results.summary())


真のATE: 2.000
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.556
Model:                            OLS   Adj. R-squared:                  0.556
Method:                 Least Squares   F-statistic:                     2502.
Date:                Thu, 21 May 2026   Prob (F-statistic):               0.00
Time:                        05:56:25   Log-Likelihood:                -3455.0
No. Observations:                2000   AIC:                             6914.
Df Residuals:                    1998   BIC:                             6925.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0285      0.044      0

# シナリオ2. 効果の異質性

In [30]:
# 真の処置効果: beta_0 + delta * X1
beta_0 = 2
delta = 1.5
u = 0.5 * X[:, 3] - 0.2 * X[:, 4] + np.random.rand(N)
Y = beta_0 * A + delta * A * X[:, 0] + 1.2 * X[:, 0] + 0.6 * X[:, 1] - 0.3 * X[:, 2] + u

true_ATE = beta_0 + delta * X[:, 0].mean()
print(f"真のATE: {true_ATE:.3f}")

data = pd.DataFrame(X[:, 0:3], columns=["X1", "X2", "X3"])
data['A'] = A
data['Y'] = Y

# (a) Linear constant
X_reg = sm.add_constant(data[['A', 'X1', 'X2', 'X3']])
beta_hat = sm.OLS(data['Y'], X_reg).fit().params['A']
print(f"Linear constantのβ_hat: {beta_hat:.3f}")

# (b) 周辺化: 単一モデル + AX交互作用 + Aを入れ替えて予測
feat_cols = ['A', 'X1', 'X2', 'X3', 'AX1']
data['AX1'] = data['A'] * data['X1']
X_train = sm.add_constant(data[feat_cols])
model1 = sm.OLS(data['Y'], X_train).fit()

# 予測用データ: Aを1/0に入れ替え、AX1も再計算
data_A1 = data.copy()
data_A1['A'] = 1
data_A1['AX1'] = data_A1['A'] * data_A1['X1']

data_A0 = data.copy()
data_A0['A'] = 0
data_A0['AX1'] = data_A0['A'] * data_A0['X1']

Y1_pred = model1.predict(sm.add_constant(data_A1[feat_cols], has_constant='add'))
Y0_pred = model1.predict(sm.add_constant(data_A0[feat_cols], has_constant='add'))
ATE_marg1 = (Y1_pred - Y0_pred).mean()
print(f"周辺化（共通モデル）: {ATE_marg1:.3f}")

# (c) 周辺化: 群別に線形回帰
Xcols = ['X1', 'X2', 'X3']
data1 = data[data['A'] == 1]
data0 = data[data['A'] == 0]
mu1 = sm.OLS(data1['Y'], sm.add_constant(data1[Xcols])).fit()
mu0 = sm.OLS(data0['Y'], sm.add_constant(data0[Xcols])).fit()
Y1_pred = model1.predict(sm.add_constant(data_A1[feat_cols], has_constant='add'))
Y0_pred = model1.predict(sm.add_constant(data_A0[feat_cols], has_constant='add'))
ATE_marg2 = (Y1_pred - Y0_pred).mean()
print(f"周辺化（群別モデル）: {ATE_marg2:.3f}")

真のATE: 2.021
Linear constantのβ_hat: 2.328
周辺化（共通モデル）: 2.427
周辺化（群別モデル）: 2.427


# シナリオ3. 効果の異質性と非線形性

In [31]:
from sklearn.ensemble import RandomForestRegressor

beta_0 = 2
delta = 1.5
u = 0.5 * X[:, 3] - 0.2 * X[:, 4] + np.random.rand(N)
Y = beta_0 * A + delta * A * X[:, 0]**2 + 1.2 * X[:, 0] + 0.6 * X[:, 1] - 0.3 * X[:, 2] + u

true_ATE = beta_0 + delta * (X[:, 0]**2).mean()
print(f"真のATE: {true_ATE:.3f}")

data = pd.DataFrame(X[:, 0:3], columns=["X1", "X2", "X3"])
data['A'] = A
data['Y'] = Y

# (a) 周辺化: 群別に線形回帰 → 非線形を捉えられずズレる
data1 = data[data['A'] == 1]
data0 = data[data['A'] == 0]
mu1 = sm.OLS(data1['Y'], sm.add_constant(data1[['X1', 'X2', 'X3']])).fit()
mu0 = sm.OLS(data0['Y'], sm.add_constant(data0[['X1', 'X2', 'X3']])).fit()
Y1_pred = mu1.predict(sm.add_constant(data[['X1', 'X2', 'X3']]))
Y0_pred = mu0.predict(sm.add_constant(data[['X1', 'X2', 'X3']]))
ATE_marg2_linear = (Y1_pred - Y0_pred).mean()
print(f"周辺化（群別モデル-線形）: {ATE_marg2_linear:.3f}")

# (b) 周辺化 + 機械学習: 群別にランダムフォレスト → 一致する
Xcols = ['X1', 'X2', 'X3']
rf1 = RandomForestRegressor(n_estimators=200, random_state=1).fit(data1[Xcols], data1['Y'])
rf0 = RandomForestRegressor(n_estimators=200, random_state=1).fit(data0[Xcols], data0['Y'])
Y1_pred = rf1.predict(data[Xcols])
Y0_pred = rf0.predict(data[Xcols])
ATE_marg2_ml = (Y1_pred - Y0_pred).mean()
print(f"周辺化（群別モデル-機械学習）: {ATE_marg2_ml:.3f}")

真のATE: 3.324
周辺化（群別モデル-線形）: 3.607
周辺化（群別モデル-機械学習）: 3.737
